# Task 5 — Classificatori con Scikit-Learn

In questo notebook addestriamo più classificatori con la libreria Scikit-Learn sul dataset
`training_clean.csv`, li confrontiamo e scegliamo il migliore. Il modello scelto sarà quello
che in sede d'esame verrà applicato al file `real_settings.csv`.

A differenza dei Task 2 e 4, dove i parametri dei modelli li calcolavamo noi a mano, qui è
la libreria a impararli dai dati con il metodo `fit()`. Il procedimento però è lo stesso che
abbiamo fatto manualmente: per esempio il Decision Tree di Scikit-Learn sceglie le soglie
provando gli split e misurando quanto separano le classi, come facevamo noi con
l'information gain nel Task 2.1.

Il lavoro è organizzato così:
1. preparazione dei dati e divisione in training set e test set;
2. un modello di riferimento (baseline) da battere;
3. primo confronto tra cinque classificatori visti a lezione;
4. gestione dello sbilanciamento delle classi;
5. validazione più robusta con la cross-validation;
6. ottimizzazione degli iperparametri con GridSearchCV;
7. scelta del modello finale e valutazione sul test set;

## 1. Caricamento e preparazione dei dati

Carichiamo il dataset pulito e separiamo:
- **X**: la matrice delle feature (le 4 colonne numeriche), cioè gli input;
- **y**: il vettore target (LABEL), cioè quello che il modello deve imparare a prevedere.

È la rappresentazione standard richiesta da Scikit-Learn (vista a lezione: feature matrix
e target array).

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv("../data/training_clean.csv")

FEATS = ["FEATURE0", "FEATURE1", "FEATURE2", "FEATURE3"]
X = df[FEATS]     # le feature (input)
y = df["LABEL"]   # il target (output da prevedere)

print("X:", X.shape)
print("y:", y.shape, "- di cui dropout:", y.sum())

X: (49799, 4)
y: (49799,) - di cui dropout: 490


## 2. Divisione in training set e test set

Il Task 4 ci ha mostrato che valutare un modello sugli stessi dati usati per costruirlo dà
risultati sballati.
Per questo dividiamo i dati in due parti:

- **training set (80%)**: i modelli imparano solo da qui;
- **test set (20%)**: righe che i modelli non vedono mai durante l'addestramento.
  Le useremo solo alla fine, per stimare come il modello scelto si comporterà su dati
  nuovi (come sarà `real_settings.csv`).

Usiamo due accorgimenti importanti:
- `stratify=y`: mantiene in entrambe le parti la stessa proporzione tra le classi (~99/1).
  Senza, il test set potrebbe ricevere per caso troppi o troppo pochi dropout;
- `random_state=42`: fissa il seed, così la divisione è riproducibile (stessa scelta fatta
  in tutti i task precedenti).

In [2]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,      # 20% al test set
    stratify=y,         # proporzione delle classi conservata
    random_state=42     # riproducibilità
)

print("train:", X_train.shape, "- dropout:", y_train.sum())
print("test: ", X_test.shape, "- dropout:", y_test.sum())

train: (39839, 4) - dropout: 392
test:  (9960, 4) - dropout: 98


La stratificazione ha funzionato: il train ha 392 dropout su 39.839 (~1%) e il test 98 su
9.960 (~1%), come il dataset di partenza.

Definiamo anche una funzione che calcola le metriche di un modello sul test set,
così non ripetiamo lo stesso codice per ogni classificatore. Usiamo le funzioni di
Scikit-Learn, che calcolano le stesse formule usate a mano nei task precedenti.

In [3]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix

def valuta_sul_test(nome, modello):
    """Addestra il modello sul training set, lo valuta sul test set
    e restituisce le metriche in un dizionario."""
    modello.fit(X_train, y_train)          # addestramento (solo sul train)
    pred = modello.predict(X_test)         # predizione sul test

    # zero_division=0: se il modello non predice mai la classe 1,
    # precision e recall valgono 0 invece di generare un errore
    return {
        "Modello": nome,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred, zero_division=0),
        "Recall": recall_score(y_test, pred, zero_division=0),
        "F1": f1_score(y_test, pred, zero_division=0),
    }

## 3. La baseline da battere

Come riferimento usiamo il classificatore banale che predice sempre la classe maggioritaria
("sempre 0"), già incontrato nel Task 4: accuracy ~99% ma recall e F1 pari a zero, perché
non trova nessun dropout.

Qualsiasi modello serio deve fare meglio di F1 = 0. Sembra un obiettivo facile, ma sui
dataset molto sbilanciati non lo è.

## 4. Primo confronto: cinque classificatori con impostazioni di default

Proviamo cinque modelli visti a lezione:

- **Gaussian Naive Bayes**: la versione automatica del classificatore del Task 2.2;
- **Decision Tree**: la versione automatica dell'albero del Task 2.1;
- **Logistic Regression**: modello lineare che stima la probabilità della classe;
- **k-Nearest Neighbors**: classifica guardando i k campioni più simili;
- **Perceptron**: modello lineare iterativo.

Nota su Pipeline e StandardScaler: la regressione logistica, il k-NN e il perceptron sono
sensibili alla scala delle feature (il k-NN ad esempio ragiona sulle distanze). Per questi
usiamo `StandardScaler`, che porta ogni feature a media 0 e deviazione standard 1.
Lo scaler va inserito in una `Pipeline` insieme al modello: così le statistiche per la
standardizzazione vengono calcolate solo sul training set, senza usare informazioni del
test set.

In [4]:
from sklearn.naive_bayes import GaussianNB
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression, Perceptron
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

modelli_default = {
    "Naive Bayes": GaussianNB(),
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Logistic Regression": Pipeline([("scaler", StandardScaler()),
                                     ("modello", LogisticRegression(max_iter=1000))]),
    "k-NN": Pipeline([("scaler", StandardScaler()),
                      ("modello", KNeighborsClassifier())]),
    "Perceptron": Pipeline([("scaler", StandardScaler()),
                            ("modello", Perceptron(random_state=42))]),
}

risultati = [valuta_sul_test(nome, m) for nome, m in modelli_default.items()]
pd.DataFrame(risultati).set_index("Modello").round(4)

,Accuracy,Precision,Recall,F1
Modello,,,,
Naive Bayes,0.9898,0.0,0.0,0.0
Decision Tree,0.9902,0.0,0.0,0.0
Logistic Regression,0.9902,0.0,0.0,0.0
k-NN,0.9902,0.0,0.0,0.0
Perceptron,0.9902,0.0,0.0,0.0


### Risultato: tutti inutili

Tutti e cinque i modelli hanno accuracy ~99% e **F1 = 0**: si sono ridotti al classificatore
banale "sempre 0" (o quasi: il Naive Bayes prova 4 predizioni positive, tutte sbagliate).

Il motivo è lo sbilanciamento: durante l'addestramento ogni errore "pesa" uguale, e siccome
i non-dropout sono il 99%, il modo più conveniente di minimizzare gli errori è ignorare
completamente la classe rara. I modelli non sono "sbagliati": stanno facendo esattamente quello
che gli è stato chiesto — solo che quello che gli è stato chiesto (minimizzare gli errori
totali) non è quello che serve a noi (trovare i dropout).

È la conferma sperimentale di quanto visto nei Task 3 e 4: su questo dataset l'accuracy
non significa nulla, e senza interventi specifici lo sbilanciamento vince.

## 5. Gestione dello sbilanciamento: class_weight

Diversi modelli di Scikit-Learn accettano il parametro `class_weight="balanced"`, che
assegna a ogni classe un peso inversamente proporzionale alla sua frequenza: nel nostro
caso sbagliare un dropout costa circa 100 volte più che sbagliare un non-dropout.
In questo modo il modello non può più "ignorare" la classe rara.

Lo applichiamo ai modelli che lo supportano (Decision Tree, Logistic Regression,
Perceptron). Il k-NN non ha questo parametro; per il Naive Bayes proviamo l'equivalente
più vicino, cioè forzare probabilità a priori uguali (50/50).

In [5]:
modelli_bilanciati = {
    "Naive Bayes (priori 50/50)": GaussianNB(priors=[0.5, 0.5]),
    "Decision Tree (balanced)": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Logistic Regression (balanced)": Pipeline([("scaler", StandardScaler()),
                                                ("modello", LogisticRegression(max_iter=1000, class_weight="balanced"))]),
    "Perceptron (balanced)": Pipeline([("scaler", StandardScaler()),
                                       ("modello", Perceptron(random_state=42, class_weight="balanced"))]),
}

risultati2 = [valuta_sul_test(nome, m) for nome, m in modelli_bilanciati.items()]
pd.DataFrame(risultati2).set_index("Modello").round(4)

,Accuracy,Precision,Recall,F1
Modello,,,,
Naive Bayes (priori 50/50),0.1965,0.0110,0.9082,0.0218
Decision Tree (balanced),0.7829,0.0253,0.5612,0.0484
Logistic Regression (balanced),0.8042,0.0276,0.5510,0.0525
Perceptron (balanced),0.8042,0.0276,0.5510,0.0525


### Risultato: i modelli "si svegliano"

Con i pesi bilanciati i modelli iniziano davvero a cercare i dropout: l'F1 passa da 0 a
circa 0.05, con recall intorno al 55%. L'accuracy scende all'80% — ed è giusto così: il
modello ora accetta dei falsi positivi pur di individuare la classe rara.

Decision Tree, Logistic Regression e Perceptron danno risultati quasi identici tra loro.
Il Naive Bayes con i priori forzati invece resta indietro: l'ipotesi di indipendenza tra
le feature e l'assunzione di distribuzione gaussiana si adattano male a feature
quasi-discrete come le nostre.

## 6. Cross-validation stratificata

Finora abbiamo valutato su un'unica divisione train/test: i numeri potrebbero dipendere
dalla fortuna di quella specifica divisione. Per una stima più affidabile usiamo la
**10-fold cross-validation stratificata** : il training set viene diviso
in 10 blocchi che rispettano le proporzioni delle classi; ogni modello viene addestrato e
valutato 10 volte (9 blocchi per addestrare, 1 per validare) e si calcolano media e
deviazione standard dell'F1.

Importante: la cross-validation usa **solo il training set**. Il test set resta da parte
per la valutazione finale.

In [6]:
from sklearn.model_selection import cross_val_score, StratifiedKFold

cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

candidati = {
    "Naive Bayes (priori 50/50)": GaussianNB(priors=[0.5, 0.5]),
    "Decision Tree (balanced)": DecisionTreeClassifier(class_weight="balanced", random_state=42),
    "Logistic Regression (balanced)": Pipeline([("scaler", StandardScaler()),
                                                ("modello", LogisticRegression(max_iter=1000, class_weight="balanced"))]),
    "Perceptron (balanced)": Pipeline([("scaler", StandardScaler()),
                                       ("modello", Perceptron(random_state=42, class_weight="balanced"))]),
}

righe = []
for nome, m in candidati.items():
    punteggi = cross_val_score(m, X_train, y_train, cv=cv, scoring="f1")
    righe.append({"Modello": nome,
                  "F1 medio": punteggi.mean(),
                  "Dev. standard": punteggi.std()})

pd.DataFrame(righe).set_index("Modello").round(4)

,F1 medio,Dev. standard
Modello,,
Naive Bayes (priori 50/50),0.0216,0.0011
Decision Tree (balanced),0.0417,0.0071
Logistic Regression (balanced),0.0432,0.0083
Perceptron (balanced),0.0258,0.0169


La cross-validation conferma il quadro: Decision Tree e Logistic Regression bilanciati sono
i migliori e sono anche stabili (deviazione standard bassa rispetto alla media). Il Naive
Bayes resta circa alla metà del loro F1.

## 7. Ottimizzazione degli iperparametri con GridSearchCV

Gli **iperparametri** sono le "manopole" che non vengono apprese dal fit ma vanno scelte da
noi: la profondità massima dell'albero.

`GridSearchCV` prova tutte le combinazioni di una griglia di valori e sceglie quella con
l'F1 medio più alto **in cross-validation sul training set** (mai sul test set).

Ottimizziamo:
- Decision Tree: `max_depth` (contro l'overfitting, come discusso nel Task 2.1) e
  `min_samples_leaf` (evita foglie con pochissimi campioni).

In [7]:
from sklearn.model_selection import GridSearchCV

# --- albero: proviamo profondità e dimensione minima delle foglie ---
griglia_albero = {
    "max_depth": [2, 3, 4, 6, 8, None],
    "min_samples_leaf": [1, 20, 100],
}
ricerca_albero = GridSearchCV(
    DecisionTreeClassifier(class_weight="balanced", random_state=42),
    griglia_albero, scoring="f1", cv=cv
)
ricerca_albero.fit(X_train, y_train)
print("Albero con migliori parametri:", ricerca_albero.best_params_,
      "F1 in cross-validation:", round(ricerca_albero.best_score_, 4))


Albero con migliori parametri: {'max_depth': 2, 'min_samples_leaf': 1} F1 in cross-validation: 0.0433


Per l'albero la combinazione migliore è `max_depth=2`: una profondità molto bassa, che
conferma quanto discusso nel Task 2.1 sul rischio di overfitting degli alberi profondi —
qui, con un segnale debole, conviene un albero piccolo.

## 8. Scelta del modello finale e valutazione sul test set
Il **Decision Tree (class_weight="balanced", max_depth=2)**:

1. produce regole esplicite e leggibili, che possiamo confrontare direttamente con il
   lavoro manuale dei Task 2 e 4;
2. non richiede standardizzazione delle feature, quindi è più semplice da applicare.

Ora usiamo il test set: la valutazione finale va fatta una volta sola,
sul modello già scelto.

In [8]:
modello_finale = DecisionTreeClassifier(class_weight="balanced", max_depth=2,
                                        min_samples_leaf=1, random_state=42)
modello_finale.fit(X_train, y_train)

pred_test = modello_finale.predict(X_test)

tn, fp, fn, tp = confusion_matrix(y_test, pred_test).ravel()
print("Confusion matrix sul test set:")
print(pd.DataFrame([[tn, fp], [fn, tp]],
                   index=["Reale 0", "Reale 1"],
                   columns=["Predetta 0", "Predetta 1"]))
print()
print(f"Accuracy : {accuracy_score(y_test, pred_test):.4f}")
print(f"Precision: {precision_score(y_test, pred_test):.4f}")
print(f"Recall   : {recall_score(y_test, pred_test):.4f}")
print(f"F1-score : {f1_score(y_test, pred_test):.4f}")

Confusion matrix sul test set:
         Predetta 0  Predetta 1
Reale 0        7960        1902
Reale 1          44          54

Accuracy : 0.8046
Precision: 0.0276
Recall   : 0.5510
F1-score : 0.0526


### Le regole imparate dal modello finale

Essendo un albero poco profondo, possiamo stamparne la struttura e leggerla.

In [9]:
from sklearn.tree import export_text

print(export_text(modello_finale, feature_names=FEATS))

|--- FEATURE1 <= 0.84
|   |--- FEATURE2 <= -0.14
|   |   |--- class: 0
|   |--- FEATURE2 >  -0.14
|   |   |--- class: 0
|--- FEATURE1 >  0.84
|   |--- FEATURE1 <= 3.38
|   |   |--- class: 1
|   |--- FEATURE1 >  3.38
|   |   |--- class: 0



Il risultato è notevole: la radice dell'albero appreso automaticamente è
**FEATURE1 <= 0.84** — praticamente la stessa regola che avevamo trovato noi nel Task 4
analizzando i tassi di dropout (FEATURE1 > 0.8365 → dropout). Scikit-Learn, partendo da
zero su 40.000 righe, è arrivato alla stessa conclusione della nostra analisi manuale:
FEATURE1 è il segnale più forte, e la soglia giusta è quella. In più l'albero aggiunge un
dettaglio che noi non avevamo visto: i valori *estremamente* alti di FEATURE1 (> 3.38)
tornano ad essere di classe 0.



## 9. Analisi critica

- **Il valore assoluto delle prestazioni è basso** (F1 ~0.05) e va detto chiaramente: non
  è un fallimento dei modelli, è il limite dell'informazione contenuta nelle feature.
  Le 4 feature assumono circa 300 combinazioni distinte per ~50.000 osservazioni, e anche
  la condizione migliore (FEATURE1 alta) porta il tasso di dropout solo dal ~1% al ~2.4%.
  Con questi ingredienti nessun algoritmo può fare miracoli — lo dimostra il fatto che
  modelli molto diversi (albero, regressione logistica, perceptron) convergono tutti sullo
  stesso F1.
- **La gestione dello sbilanciamento è stata la scelta più importante di tutto il task**:
  senza `class_weight` tutti i modelli degeneravano nel "sempre 0" (F1=0). Nessun tuning
  di iperparametri avrebbe potuto compensare questa mancanza.